# 12. v2 데이터 만들기 — 계절 5장 (고려대 PCF 방식) 내보내기 + 캐시

**한 줄 요약:** 지역마다 3–4·5–6·7–8·9–10·11–12월 중앙값 × 4밴드(B2·B3·B4·B8) = 20층을 GEE로 내보내고, **v1 캐시와 같은 픽셀 위치**에서 값을 뽑아 `{지역}_v2.npz`로 저장한 뒤 원본 영상을 지운다.

**쉬운 설명**
- 고려대 논문(Kim et al. 2025)과 같은 계절 구성: 2개월 간격 5장, 1–2월 제외(눈)
- 격자는 v1 영상과 **완전히 같게** 내보낸다 → v1 캐시의 픽셀과 1:1로 짝지을 수 있음 (실험 6에서 "v1 + 계절" 비교용)
- Drive 용량이 부족해서 **2곳씩** 처리: 내보내기 → 캐시 → 원본 삭제 → 다음 2곳

**v1과 같게 둔 것:** 기간 2024–2025, Sentinel-2 SR Harmonized, SCL 구름 마스크(3·8·9·10 제외), 구름 50% 미만 장면, int16(×10000)
**고려대와 다른 것:** 그들은 QA60·구름 30% 미만, 연도별. 우리는 v1과 맞추려고 SCL·50%·2년 합침

**v1 영상(`{지역}_s2_20band.tif`)은 지우지 않는다.** 픽셀 위치를 재현하는 데 필요하다.

**순서:** [1] 설정 → [2] 함수 → [3] 용량 확인 → **묶음 A~D를 위에서부터 차례로** → [7] 진행표

각 묶음은 같은 모양이다: **① 제출 → ② 상태 확인(COMPLETED까지 반복) → ③ 이동·캐시·원본 삭제 → ④ Drive 휴지통 비우기**. 지역 이름이 이미 들어가 있으니 고칠 것 없이 실행만 하면 된다.

런타임이 끊기면 [1]·[2]만 다시 실행하고 하던 묶음으로 돌아오면 된다. 캐시가 이미 있는 지역은 자동으로 건너뛴다.

**삭제 주의:** Colab에서 지운 파일은 Drive **휴지통**으로 가서 용량을 계속 차지한다. 2곳 끝날 때마다 Drive 휴지통을 비울 것.


In [1]:
# [1] 설정 — 02 [1]과 같은 GEE 프로젝트·경로
!pip install -q earthengine-api rasterio

import ee, os, glob, json, time, gc, shutil
import numpy as np
import rasterio
from rasterio.windows import Window
from scipy.ndimage import distance_transform_edt
from google.colab import drive

PROJECT_ID = 'driven-era-467023-m3'
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT_ID)

drive.mount('/content/drive')
D       = '/content/drive/MyDrive/forest/data'
OUT     = '/content/drive/MyDrive/forest/outputs'
SRC_DIR = '/content/drive/MyDrive/forest_s2'          # GEE가 내보내는 곳 (02와 같음)
LOG     = f'{OUT}/v2_export_log.json'

TRAIN_REGIONS = ['inje', 'hongcheon', 'gangneung', 'pyeongchang',
                 'bonghwa', 'geumsan', 'geochang', 'suncheon']

START, END = '2024-01-01', '2025-12-31'               # v1과 같음
SEASONS  = [('s1', [3, 4]), ('s2', [5, 6]), ('s3', [7, 8]), ('s4', [9, 10]), ('s5', [11, 12])]   # 고려대 PCF
BANDS_V2 = ['B2', 'B3', 'B4', 'B8']                   # 고려대: 가시광 + 근적외
NODATA   = -1                                         # 구름 때문에 값이 없는 픽셀

# v1 캐시 재현용 (05 extract_samples와 같은 값)
N_MAX, BLOCK, SEED_SAMPLE, TILE, EDGE_PX = 600_000, 1024, 0, 500, 7

DELETE_RAW = True     # 캐시 저장·검산 후 v2 원본 영상 삭제 (GEE로 언제든 다시 만들 수 있음)

for r in TRAIN_REGIONS:
    assert os.path.exists(f'{D}/s2/{r}_s2_20band.tif'), f'v1 영상 없음: {r} (지우면 안 됨)'
    assert os.path.exists(f'{OUT}/{r}_s05.npz'), f'v1 캐시 없음: {r}'
print('v1 영상·캐시 8곳 확인')


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Mounted at /content/drive
v1 영상·캐시 8곳 확인


In [2]:
# [2] 함수
def cloud_mask(img):
    # 02와 같음: SCL 3(그림자) 8·9(구름) 10(권운) 제외
    scl = img.select('SCL')
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good).divide(10000)

def season_composite(aoi, months, cloud_pct=50):
    mf = ee.Filter.Or([ee.Filter.calendarRange(m, m, 'month') for m in months])
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(aoi).filterDate(START, END).filter(mf)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_pct)))
    return col.map(cloud_mask).median(), col.size()

def v1_grid(name):
    with rasterio.open(f'{D}/s2/{name}_s2_20band.tif') as s:
        epsg = s.crs.to_epsg()
        assert epsg is not None, f'{name}: v1 좌표계 EPSG 코드 없음 — 멈추고 보고'
        return dict(crs=f'EPSG:{epsg}', tr=s.transform, b=s.bounds, H=s.height, W=s.width)

def submit_v2(name):
    g = v1_grid(name)
    aoi = ee.Geometry.Rectangle([g['b'].left, g['b'].bottom, g['b'].right, g['b'].top],
                                proj=g['crs'], geodesic=False)
    imgs, counts = [], {}
    for tag, months in SEASONS:
        img, n = season_composite(aoi, months)
        imgs.append(img.select(BANDS_V2).rename([f'{tag}_{b}' for b in BANDS_V2]))
        counts[tag] = int(n.getInfo())
    stack = ee.Image.cat(imgs).multiply(10000).round().toInt16().unmask(NODATA)
    tr = g['tr']
    task = ee.batch.Export.image.toDrive(
        image=stack, description=f'{name}_s2v2', folder='forest_s2',
        fileNamePrefix=f'{name}_s2v2_20band', region=aoi, crs=g['crs'],
        crsTransform=[tr.a, tr.b, tr.c, tr.d, tr.e, tr.f], maxPixels=1e10)
    task.start()
    log = json.load(open(LOG)) if os.path.exists(LOG) else {}
    log[name] = dict(submitted=time.strftime('%Y-%m-%d %H:%M'), scenes=counts, crs=g['crs'], H=g['H'], W=g['W'])
    json.dump(log, open(LOG, 'w'), ensure_ascii=False, indent=2)
    print(f'{name:12s} 제출  장면 수 {counts}  격자 {g["H"]}x{g["W"]} {g["crs"]}')

def image_ok_mask(src, H, W, block=2048, edge=EDGE_PX):
    # 05와 같음 (v1 영상 기준) — 픽셀 위치 재현에 필요
    valid = np.zeros((H, W), bool)
    for r0 in range(0, H, block):
        for c0 in range(0, W, block):
            r1, c1 = min(H, r0 + block), min(W, c0 + block)
            X = src.read(window=Window(c0, r0, c1 - c0, r1 - r0))
            valid[r0:r1, c0:c1] = X.sum(axis=0, dtype='float32') > 0
            del X
    if valid.all():
        return valid
    return distance_transform_edt(valid) >= edge

def build_v2_cache(name, v2_path):
    # 05 extract_samples와 같은 순서·같은 난수로 픽셀을 다시 골라, 그 위치의 v2 값을 뽑는다
    t0 = time.time()
    with rasterio.open(f'{D}/{name}_label.tif') as s: label = s.read(1)
    with rasterio.open(f'{D}/{name}_year.tif')  as s: year  = s.read(1)
    H, W = label.shape
    use0 = (label >= 1) & (label <= 3) & (year >= 2024)
    del year
    with rasterio.open(f'{D}/s2/{name}_s2_20band.tif') as src1:
        assert (src1.height, src1.width) == (H, W)
        t1, crs1 = src1.transform, src1.crs
        use = use0 & image_ok_mask(src1, H, W)
    n_total = int(use.sum()); del use0; gc.collect()

    with rasterio.open(v2_path) as src2:
        t2 = src2.transform
        assert src2.crs == crs1, f'{name}: v2 좌표계 다름'
        assert src2.count == 20, f'{name}: v2 밴드 수 {src2.count}'
        assert abs(t2.a - t1.a) < 1e-6 and abs(t2.e - t1.e) < 1e-6, f'{name}: 픽셀 크기 다름'
        dc, dr = (t1.c - t2.c) / t1.a, (t1.f - t2.f) / t1.e
        assert abs(dc - round(dc)) < 1e-3 and abs(dr - round(dr)) < 1e-3, f'{name}: 격자 어긋남 {dc:.3f},{dr:.3f}'
        co, ro = int(round(dc)), int(round(dr))

        frac  = min(1.0, N_MAX / max(n_total, 1))
        rng_b = np.random.default_rng(SEED_SAMPLE)
        V, ys, ts, rows, cols = [], [], [], [], []
        for r0 in range(0, H, BLOCK):
            for c0 in range(0, W, BLOCK):
                r1, c1 = min(H, r0 + BLOCK), min(W, c0 + BLOCK)
                m = use[r0:r1, c0:c1]
                if not m.any(): continue
                sel = m & (rng_b.random(m.shape) < frac)
                if not sel.any(): continue
                rr, cc = np.where(sel)
                X = src2.read(window=Window(c0 + co, r0 + ro, c1 - c0, r1 - r0), boundless=True, fill_value=NODATA)
                V.append(X[:, sel].T)
                ys.append(label[r0:r1, c0:c1][sel])
                ts.append(((rr + r0) // TILE) * 10000 + (cc + c0) // TILE)
                rows.append(rr + r0); cols.append(cc + c0)
                del X

    V = np.concatenate(V); y = np.concatenate(ys).astype('int8'); tile = np.concatenate(ts).astype('int32')
    # 검산: v1 캐시와 같은 픽셀인가 (개수·라벨·타일 순서가 전부 같아야 함)
    z = np.load(f'{OUT}/{name}_s05.npz')
    assert len(y) == len(z['y']), f'{name}: 표본 수 {len(y)} ≠ v1 {len(z["y"])} — 멈추고 보고'
    assert (y == z['y']).all() and (tile == z['tile']).all(), f'{name}: 픽셀 순서가 v1과 다름 — 멈추고 보고'
    assert int(z['n_total']) == n_total, f'{name}: n_total 다름'

    ok_s = np.stack([(V[:, 4 * k:4 * k + 4] > 0).all(1) for k in range(5)], 1)   # 계절별 값 있음
    ok = ok_s.all(1)
    B = V.astype('float32') / 10000.0
    nd = lambda a, b: (a - b) / (a + b + 1e-6)
    ndvi = np.stack([nd(B[:, 4 * k + 3], B[:, 4 * k + 2]) for k in range(5)], 1)   # B8, B4
    X2 = np.concatenate([B, ndvi], 1).astype('float32')
    X2[~ok] = np.nan
    names = [f'{t}_{b}' for t, _ in SEASONS for b in BANDS_V2] + [f'{t}_ndvi' for t, _ in SEASONS]
    np.savez_compressed(f'{OUT}/{name}_v2.npz', X2=X2, ok=ok, ok_season=ok_s, y=y, tile=tile,
                        rows=np.concatenate(rows).astype('int32'), cols=np.concatenate(cols).astype('int32'),
                        n_total=np.int64(n_total), names=np.array(names))
    # 저장 확인
    zz = np.load(f'{OUT}/{name}_v2.npz')
    assert zz['X2'].shape == (len(y), 25)
    miss = (~ok_s).mean(0) * 100
    print(f'{name:12s} 캐시 저장 {len(y):,}픽셀, 값 있음 {ok.mean()*100:.1f}%  '
          f'계절별 결측% ' + ' '.join(f'{t}:{m:.1f}' for (t, _), m in zip(SEASONS, miss)) + f'  ({time.time()-t0:.0f}초)')
    return ok.mean()

# ---- 묶음 실행용 (원래 [4]·[5]·[6] 셀 내용을 지역 지정 함수로 옮긴 것, 동작 같음) ----
def submit_pair(now):
    for r in now:
        if os.path.exists(f'{OUT}/{r}_v2.npz'):
            print(f'{r:12s} 캐시 있음 — 건너뜀'); continue
        if glob.glob(f'{SRC_DIR}/{r}_s2v2_20band*.tif') or glob.glob(f'{D}/s2/{r}_s2v2_20band*.tif'):
            print(f'{r:12s} 영상이 이미 Drive에 있음 — ③으로'); continue
        submit_v2(r)
    print('\n→ ② 상태 확인 (지역당 수십 분 [추정])')

def status_pair(now):
    want = {f'{r}_s2v2' for r in now}
    seen = set()
    for t in ee.batch.Task.list()[:30]:
        d = t.status()
        desc = d.get('description', '')
        if desc in want and desc not in seen:      # 같은 이름이 여러 번이면 가장 최근 것만
            seen.add(desc)
            print(f"{desc:22s} {d['state']:12s} {d.get('error_message', '')}")
    for r in now:
        if os.path.exists(f'{OUT}/{r}_v2.npz'):
            print(f'{r:12s} 캐시 이미 있음')
    print('\n둘 다 COMPLETED → ③ / 아직이면 몇 분 뒤 이 셀 다시 실행')

def process_pair(now):
    os.makedirs(f'{D}/s2', exist_ok=True)
    for r in now:
        if os.path.exists(f'{OUT}/{r}_v2.npz'):
            print(f'{r:12s} 캐시 있음 — 건너뜀'); continue
        parts = sorted(glob.glob(f'{SRC_DIR}/{r}_s2v2_20band*.tif')) + sorted(glob.glob(f'{D}/s2/{r}_s2v2_20band*.tif'))
        if not parts:
            print(f'{r:12s} 영상이 아직 Drive에 안 보임 — 몇 분 뒤 이 셀 다시 실행'); continue
        assert len(parts) == 1, f'{r}: 파일이 여러 조각으로 나뉨 {parts} — 멈추고 보고'
        p = parts[0]
        if p.startswith(SRC_DIR):
            dst = f'{D}/s2/{os.path.basename(p)}'
            shutil.move(p, dst); p = dst
        print(f'{r:12s} 영상 {os.path.getsize(p)/1e9:.2f} GB')
        build_v2_cache(r, p)
        if DELETE_RAW:
            os.remove(p)
            print(f'{r:12s} v2 원본 삭제 (Drive 휴지통 비우기 필요)')
    done = [r for r in now if os.path.exists(f'{OUT}/{r}_v2.npz')]
    print(f'\n이 묶음 {len(done)}/{len(now)} 완료' + ('  → ④ 휴지통 비우고 다음 묶음' if len(done) == len(now) else ''))

print('함수 정의 완료')


함수 정의 완료


In [3]:
# [3] 용량 확인 — v1 영상 크기를 보고 v2 한 곳이 얼마나 될지 가늠
tot, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive: 전체 {tot/1e9:.1f} GB / 사용 {used/1e9:.1f} GB / 남음 {free/1e9:.1f} GB  (Colab 표시값, 휴지통 포함 여부는 Drive 화면 기준으로 확인)')
sz = {r: os.path.getsize(f'{D}/s2/{r}_s2_20band.tif') / 1e9 for r in TRAIN_REGIONS}
for r in TRAIN_REGIONS:
    done = '캐시 있음' if os.path.exists(f'{OUT}/{r}_v2.npz') else ''
    print(f'  {r:12s} v1 영상 {sz[r]:.2f} GB  {done}')
print(f'v2는 같은 20층 int16이라 비슷한 크기 [추정]. 2곳 동시 최대 약 {sum(sorted(sz.values())[-2:]):.1f} GB 필요')


Drive: 전체 16.1 GB / 사용 11.4 GB / 남음 4.7 GB  (Colab 표시값, 휴지통 포함 여부는 Drive 화면 기준으로 확인)
  inje         v1 영상 0.48 GB  
  hongcheon    v1 영상 1.60 GB  
  gangneung    v1 영상 0.47 GB  
  pyeongchang  v1 영상 0.49 GB  
  bonghwa      v1 영상 0.49 GB  
  geumsan      v1 영상 0.37 GB  
  geochang     v1 영상 0.50 GB  
  suncheon     v1 영상 0.58 GB  
v2는 같은 20층 int16이라 비슷한 크기 [추정]. 2곳 동시 최대 약 2.2 GB 필요


---
## 묶음 A — 인제·홍천

In [4]:
# A-① 내보내기 제출 — 인제·홍천
NOW_A = ['inje', 'hongcheon']
submit_pair(NOW_A)

inje         제출  장면 수 {'s1': 52, 's2': 37, 's3': 42, 's4': 36, 's5': 74}  격자 3528x3519 EPSG:5179
hongcheon    제출  장면 수 {'s1': 76, 's2': 58, 's3': 62, 's4': 60, 's5': 112}  격자 4420x9309 EPSG:5179

→ ② 상태 확인 (지역당 수십 분 [추정])


In [13]:
# A-② 상태 확인 — 수시로 다시 실행. 둘 다 COMPLETED가 되면 A-③
status_pair(NOW_A)

hongcheon_s2v2         COMPLETED    
inje_s2v2              COMPLETED    

둘 다 COMPLETED → ③ / 아직이면 몇 분 뒤 이 셀 다시 실행


In [14]:
# A-③ 이동 → 캐시 → v2 원본 삭제
#   (GEE가 끝나도 Drive에 파일이 보이기까지 몇 분 걸릴 수 있음)
process_pair(NOW_A)

inje         영상 0.49 GB
inje         캐시 저장 600,263픽셀, 값 있음 100.0%  계절별 결측% s1:0.0 s2:0.0 s3:0.0 s4:0.0 s5:0.0  (39초)
inje         v2 원본 삭제 (Drive 휴지통 비우기 필요)
hongcheon    영상 1.66 GB
hongcheon    캐시 저장 599,525픽셀, 값 있음 100.0%  계절별 결측% s1:0.0 s2:0.0 s3:0.0 s4:0.0 s5:0.0  (104초)
hongcheon    v2 원본 삭제 (Drive 휴지통 비우기 필요)

이 묶음 2/2 완료  → ④ 휴지통 비우고 다음 묶음


**A-④ Drive 휴지통 비우기** — drive.google.com → 휴지통 → 비우기. 지운 v2 원본이 여기서 용량을 계속 차지한다. 비운 뒤 다음 묶음으로.

---
## 묶음 B — 강릉·평창
 앞 묶음(인제·홍천)의 ③이 끝나고 휴지통을 비운 뒤 시작.

In [15]:
# B-① 내보내기 제출 — 강릉·평창
NOW_B = ['gangneung', 'pyeongchang']
submit_pair(NOW_B)

gangneung    제출  장면 수 {'s1': 110, 's2': 78, 's3': 98, 's4': 78, 's5': 157}  격자 3549x3557 EPSG:5179
pyeongchang  제출  장면 수 {'s1': 26, 's2': 17, 's3': 20, 's4': 20, 's5': 34}  격자 3537x3552 EPSG:5179

→ ② 상태 확인 (지역당 수십 분 [추정])


In [20]:
# B-② 상태 확인 — 수시로 다시 실행. 둘 다 COMPLETED가 되면 B-③
status_pair(NOW_B)

pyeongchang_s2v2       COMPLETED    
gangneung_s2v2         COMPLETED    

둘 다 COMPLETED → ③ / 아직이면 몇 분 뒤 이 셀 다시 실행


In [21]:
# B-③ 이동 → 캐시 → v2 원본 삭제
#   (GEE가 끝나도 Drive에 파일이 보이기까지 몇 분 걸릴 수 있음)
process_pair(NOW_B)

gangneung    영상 0.50 GB
gangneung    캐시 저장 600,079픽셀, 값 있음 100.0%  계절별 결측% s1:0.0 s2:0.0 s3:0.0 s4:0.0 s5:0.0  (38초)
gangneung    v2 원본 삭제 (Drive 휴지통 비우기 필요)
pyeongchang  영상 0.51 GB
pyeongchang  캐시 저장 600,530픽셀, 값 있음 100.0%  계절별 결측% s1:0.0 s2:0.0 s3:0.0 s4:0.0 s5:0.0  (34초)
pyeongchang  v2 원본 삭제 (Drive 휴지통 비우기 필요)

이 묶음 2/2 완료  → ④ 휴지통 비우고 다음 묶음


**B-④ Drive 휴지통 비우기** — drive.google.com → 휴지통 → 비우기. 지운 v2 원본이 여기서 용량을 계속 차지한다. 비운 뒤 다음 묶음으로.

---
## 묶음 C — 봉화·금산
 앞 묶음(강릉·평창)의 ③이 끝나고 휴지통을 비운 뒤 시작.

In [22]:
# C-① 내보내기 제출 — 봉화·금산
NOW_C = ['bonghwa', 'geumsan']
submit_pair(NOW_C)

bonghwa      제출  장면 수 {'s1': 85, 's2': 59, 's3': 77, 's4': 56, 's5': 109}  격자 3551x3595 EPSG:5179
geumsan      제출  장면 수 {'s1': 25, 's2': 16, 's3': 30, 's4': 19, 's5': 42}  격자 3262x2892 EPSG:5179

→ ② 상태 확인 (지역당 수십 분 [추정])


In [30]:
# C-② 상태 확인 — 수시로 다시 실행. 둘 다 COMPLETED가 되면 C-③
status_pair(NOW_C)

geumsan_s2v2           COMPLETED    
bonghwa_s2v2           COMPLETED    

둘 다 COMPLETED → ③ / 아직이면 몇 분 뒤 이 셀 다시 실행


In [31]:
# C-③ 이동 → 캐시 → v2 원본 삭제
#   (GEE가 끝나도 Drive에 파일이 보이기까지 몇 분 걸릴 수 있음)
process_pair(NOW_C)

bonghwa      영상 0.52 GB
bonghwa      캐시 저장 599,540픽셀, 값 있음 99.2%  계절별 결측% s1:0.0 s2:0.0 s3:0.0 s4:0.0 s5:0.8  (38초)
bonghwa      v2 원본 삭제 (Drive 휴지통 비우기 필요)
geumsan      영상 0.38 GB
geumsan      캐시 저장 600,247픽셀, 값 있음 100.0%  계절별 결측% s1:0.0 s2:0.0 s3:0.0 s4:0.0 s5:0.0  (30초)
geumsan      v2 원본 삭제 (Drive 휴지통 비우기 필요)

이 묶음 2/2 완료  → ④ 휴지통 비우고 다음 묶음


**C-④ Drive 휴지통 비우기** — drive.google.com → 휴지통 → 비우기. 지운 v2 원본이 여기서 용량을 계속 차지한다. 비운 뒤 다음 묶음으로.

---
## 묶음 D — 거창·순천
 앞 묶음(봉화·금산)의 ③이 끝나고 휴지통을 비운 뒤 시작.

In [32]:
# D-① 내보내기 제출 — 거창·순천
NOW_D = ['geochang', 'suncheon']
submit_pair(NOW_D)

geochang     제출  장면 수 {'s1': 50, 's2': 36, 's3': 66, 's4': 38, 's5': 73}  격자 3513x3612 EPSG:5179
suncheon     제출  장면 수 {'s1': 51, 's2': 35, 's3': 67, 's4': 43, 's5': 71}  격자 3924x3758 EPSG:5179

→ ② 상태 확인 (지역당 수십 분 [추정])


In [38]:
# D-② 상태 확인 — 수시로 다시 실행. 둘 다 COMPLETED가 되면 D-③
status_pair(NOW_D)

suncheon_s2v2          COMPLETED    
geochang_s2v2          COMPLETED    

둘 다 COMPLETED → ③ / 아직이면 몇 분 뒤 이 셀 다시 실행


In [39]:
# D-③ 이동 → 캐시 → v2 원본 삭제
#   (GEE가 끝나도 Drive에 파일이 보이기까지 몇 분 걸릴 수 있음)
process_pair(NOW_D)

geochang     영상 0.51 GB
geochang     캐시 저장 601,114픽셀, 값 있음 100.0%  계절별 결측% s1:0.0 s2:0.0 s3:0.0 s4:0.0 s5:0.0  (46초)
geochang     v2 원본 삭제 (Drive 휴지통 비우기 필요)
suncheon     영상 0.59 GB
suncheon     캐시 저장 600,845픽셀, 값 있음 100.0%  계절별 결측% s1:0.0 s2:0.0 s3:0.0 s4:0.0 s5:0.0  (44초)
suncheon     v2 원본 삭제 (Drive 휴지통 비우기 필요)

이 묶음 2/2 완료  → ④ 휴지통 비우고 다음 묶음


**D-④ Drive 휴지통 비우기** — drive.google.com → 휴지통 → 비우기. 지운 v2 원본이 여기서 용량을 계속 차지한다.

---
## 마무리

In [40]:
# [7] 진행표
log = json.load(open(LOG)) if os.path.exists(LOG) else {}
print(f'{"지역":12s} {"캐시":6s} {"값있음%":>8s}  장면 수(계절별)')
for r in TRAIN_REGIONS:
    f = f'{OUT}/{r}_v2.npz'
    if os.path.exists(f):
        z = np.load(f); okp = f"{z['ok'].mean()*100:.1f}"
    else:
        okp = '-'
    sc = log.get(r, {}).get('scenes', '-')
    st = '있음' if os.path.exists(f) else '없음'
    print(f'{r:12s} {st:6s} {okp:>8s}  {sc}')
n_done = sum(os.path.exists(f'{OUT}/{r}_v2.npz') for r in TRAIN_REGIONS)
print(f'\n{n_done}/8 완료' + ('  → 8곳 끝나면 [7] 출력을 채팅에 붙여넣기' if n_done < 8 else '  → 이 출력을 채팅에 붙여넣기 (실험 6 노트북으로)'))


지역           캐시         값있음%  장면 수(계절별)
inje         있음        100.0  {'s1': 52, 's2': 37, 's3': 42, 's4': 36, 's5': 74}
hongcheon    있음        100.0  {'s1': 76, 's2': 58, 's3': 62, 's4': 60, 's5': 112}
gangneung    있음        100.0  {'s1': 110, 's2': 78, 's3': 98, 's4': 78, 's5': 157}
pyeongchang  있음        100.0  {'s1': 26, 's2': 17, 's3': 20, 's4': 20, 's5': 34}
bonghwa      있음         99.2  {'s1': 85, 's2': 59, 's3': 77, 's4': 56, 's5': 109}
geumsan      있음        100.0  {'s1': 25, 's2': 16, 's3': 30, 's4': 19, 's5': 42}
geochang     있음        100.0  {'s1': 50, 's2': 36, 's3': 66, 's4': 38, 's5': 73}
suncheon     있음        100.0  {'s1': 51, 's2': 35, 's3': 67, 's4': 43, 's5': 71}

8/8 완료  → 이 출력을 채팅에 붙여넣기 (실험 6 노트북으로)
